<a href="https://colab.research.google.com/github/FaridRash/brain-ct-hemorrhage-segmentation/blob/main/Notebooks/06_training_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

source_zip_path = '/content/drive/MyDrive/brain_ct_project/windowed_normalized_data.zip'
destination_dir = '/content'

os.makedirs(destination_dir, exist_ok=True)
!unzip -q "{source_zip_path}" -d "{destination_dir}"
print(f"'{source_zip_path}' unzipped to '{destination_dir}' successfully.")

'/content/drive/MyDrive/brain_ct_project/windowed_normalized_data.zip' unzipped to '/content' successfully.


#Data verification

In [3]:
import os
import pandas as pd
import numpy as np

base_path = "/content/brain_ct_project"

train_csv = os.path.join(base_path, "label_train.csv")
train_dir = os.path.join(base_path, "train/images")

df = pd.read_csv(train_csv)

print("Number of samples (CSV):", len(df))
print("Number of image files:", len(os.listdir(train_dir)))

print("\n--- First 5 CSV entries ---")
print(df.head())

print("\n--- First 10 image files ---")
files = sorted(os.listdir(train_dir))
print(files[:10])

# try loading ONE sample safely
sample_file = df.iloc[0]["filename"]
sample_path = os.path.join(train_dir, sample_file)

print("\nTrying to load:", sample_file)

if sample_file.endswith(".npy"):
    img = np.load(sample_path)
elif sample_file.endswith(".png"):
    import cv2
    img = cv2.imread(sample_path, cv2.IMREAD_GRAYSCALE)
else:
    raise ValueError("Unknown file format")

print("Shape:", img.shape)
print("Min:", img.min(), "Max:", img.max())

Number of samples (CSV): 2116
Number of image files: 2116

--- First 5 CSV entries ---
          filename  label
0  slice_00000.npy      0
1  slice_00001.npy      0
2  slice_00002.npy      0
3  slice_00003.npy      0
4  slice_00004.npy      0

--- First 10 image files ---
['slice_00000.npy', 'slice_00001.npy', 'slice_00002.npy', 'slice_00003.npy', 'slice_00004.npy', 'slice_00005.npy', 'slice_00006.npy', 'slice_00007.npy', 'slice_00008.npy', 'slice_00009.npy']

Trying to load: slice_00000.npy
Shape: (512, 512)
Min: 0.0 Max: 1.0


#Dataset Class

In [4]:
import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset

class BrainCTDataset(Dataset):
    def __init__(self, image_dir, csv_file):
        self.image_dir = image_dir
        self.df = pd.read_csv(csv_file)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = os.path.join(self.image_dir, row["filename"])
        image = np.load(img_path)  # already normalized

        # convert to tensor + add channel dim
        image = torch.tensor(image, dtype=torch.float32).unsqueeze(0)

        label = torch.tensor(row["label"], dtype=torch.long)

        return image, label

#Create Datasets & Loaders

In [5]:
from torch.utils.data import DataLoader

base_path = "/content/brain_ct_project"

train_dataset = BrainCTDataset(
    image_dir=os.path.join(base_path, "train/images"),
    csv_file=os.path.join(base_path, "label_train.csv")
)

val_dataset = BrainCTDataset(
    image_dir=os.path.join(base_path, "val/images"),
    csv_file=os.path.join(base_path, "label_val.csv")
)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

#Sanity Check Loader

In [6]:
images, labels = next(iter(train_loader))

print("Batch shape:", images.shape)
print("Labels shape:", labels.shape)

Batch shape: torch.Size([16, 1, 512, 512])
Labels shape: torch.Size([16])


#CNN

In [7]:
import torch.nn as nn
import torch.nn.functional as F

class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(1, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, 3, padding=1)

        self.pool = nn.MaxPool2d(2, 2)

        # 🔥 KEY PART
        self.adaptive_pool = nn.AdaptiveAvgPool2d((1, 1))

        self.fc1 = nn.Linear(64, 32)
        self.fc2 = nn.Linear(32, 2)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))   # 512 → 256
        x = self.pool(F.relu(self.conv2(x)))   # 256 → 128
        x = self.pool(F.relu(self.conv3(x)))   # 128 → 64

        x = self.adaptive_pool(x)              # → (64,1,1)
        x = torch.flatten(x, 1)                # → (64)

        x = F.relu(self.fc1(x))
        x = self.fc2(x)

        return x

#Loss

In [8]:
import numpy as np
import torch

labels = train_dataset.df["label"].values
class_counts = np.bincount(labels)

weights = 1.0 / class_counts
weights = torch.tensor(weights, dtype=torch.float32)

criterion = nn.CrossEntropyLoss(weight=weights)

#Model + Optimizer

In [9]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SimpleCNN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

#TRAIN

In [13]:
epochs = 5

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        # forward
        outputs = model(images)
        loss = criterion(outputs, labels)

        # backward
        optimizer.zero_grad()
        loss.backward()

        # 🔥 optional but recommended (stability)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        total_loss += loss.item()

    # ✅ average loss (important!)
    avg_loss = total_loss / len(train_loader)

    print(f"Epoch [{epoch+1}/{epochs}] - Loss: {avg_loss:.4f}")

Epoch [1/5] - Loss: 0.6886
Epoch [2/5] - Loss: 0.6859
Epoch [3/5] - Loss: 0.6838
Epoch [4/5] - Loss: 0.6792
Epoch [5/5] - Loss: 0.6725


#Validation Code

In [15]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)

        outputs = model(images)
        preds = torch.argmax(outputs, dim=1).cpu().numpy()

        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

# metrics
acc = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds, zero_division=0)
recall = recall_score(all_labels, all_preds, zero_division=0)
f1 = f1_score(all_labels, all_preds, zero_division=0)

print(f"Accuracy : {acc:.4f}")
print(f"Precision: {precision:.20f}")
print(f"Recall   : {recall:.20f}")
print(f"F1 Score : {f1:.20f}")

Accuracy : 0.8527
Precision: 0.00000000000000000000
Recall   : 0.00000000000000000000
F1 Score : 0.00000000000000000000


#Save the model

In [16]:
torch.save(model.state_dict(), "/content/brain_ct_model.pth")

In [17]:
import torch

save_path = "/content/brain_ct_model.pth"
torch.save(model.state_dict(), save_path)

print("Model saved at:", save_path)

Model saved at: /content/brain_ct_model.pth


In [18]:
drive_path = "/content/drive/MyDrive/brain_ct_project/brain_ct_model.pth"
torch.save(model.state_dict(), drive_path)

print("Saved to Drive:", drive_path)

Saved to Drive: /content/drive/MyDrive/brain_ct_project/brain_ct_model.pth
